# 使用Raw OpenAI API进行工具调用

## 0.环境配置

In [1]:
from dotenv import load_dotenv
from pathlib import Path
import os

env_path = Path.cwd() / ".env"

if not env_path.exists():
    env_path = Path.cwd().parent / ".env"

load_dotenv(dotenv_path=env_path, override=True)

api_key = os.getenv("DEEPSEEK_API_KEY")

print("使用的 .env 路径:", env_path)
print("是否读到 key:", bool(api_key))

使用的 .env 路径: c:\Users\ASUS\Desktop\llm_jupyter_env\.env
是否读到 key: True


## 1. 定义工具函数

In [2]:
import math

def calculator(expression: str):
    allowed = {
        "sin": math.sin,
        "cos": math.cos,
        "sqrt": math.sqrt,
        "pi": math.pi
    }
    result = eval(expression, {"__builtins__": {}}, allowed)

    return f"{expression} = {result}"

def get_weather(city: str):
    weather_data = {
        "汕头": {"天气": "晴", "温度": "28C"},
        "深圳": {"天气": "多云", "温度": "21C"},
        "上海": {"天气": "阵雨", "温度": "18C"},
        "北京": {"天气": "晴", "温度": "30C"},
    }

    if city in weather_data:
        d = weather_data[city]
        return f"{city}: {d["天气"]} 温度: {d["温度"]}"

    return "找不到所在城市的温度。"

def get_location():
    import random
    cities = ["汕头", "深圳", "上海", "北京"]

    return random.choice(cities)

tool_functions = {
    "calculator": calculator,
    "get_weather": get_weather,
    "get_location": get_location
}


## 2. 定义Tool Schema

In [3]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "可以执行简单的数学计算",
            "parameters":{
                "type": "object",
                "properties":{
                    "expression": {
                        "type": "string",
                        "description": "数学表达式"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "根据城市查询天气",
            "parameters":{
                "type": "object",
                "properties":{
                    "city": {
                        "type": "string",
                        "description": "城市名称"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_location",
            "description": "获取用户所在城市",
            "parameters":{
                "type": "object",
                "properties":{},
                "required": []
            }
        }
    }
]

## 3. 通过OpenAI API调用工具

In [4]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

messages = [
    {"role": "system", "content": "你是一个智能助手，你可以使用工具回答问题。"},
    {"role": "user", "content": "今天天气怎么样？以及2的10次方是多少？"}
]

response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

print("Response: ", response.choices[0].message.content)
print("Tool call: ", response.choices[0].message.tool_calls)

Response:  好的！我先查一下你的所在城市，同时计算2的10次方。
Tool call:  [ChatCompletionMessageFunctionToolCall(id='call_00_3IuDAMS3sKhtpEhvWHXl3654', function=Function(arguments='{}', name='get_location'), type='function', index=0), ChatCompletionMessageFunctionToolCall(id='call_01_GUQhbdegzhqzf2pBa8ff6198', function=Function(arguments='{"expression": "2^10"}', name='calculator'), type='function', index=1)]


### 3.1 执行工具并且返回结果

In [5]:
tool_calls = response.choices[0].message.tool_calls
tool_results = []

if tool_calls:
    for tool_call in tool_calls:
        func_name = tool_call.function.name
        func_args = eval(tool_call.function.arguments)
        print(f"调用: {func_name}, 参数: {func_args}")

        if func_name in tool_functions:
            result = tool_functions[func_name](**func_args) if func_args else tool_functions[func_name]()
        else:
            result = f"工具{func_name}不存在"

        print(f"结果: {result}")
        tool_results.append({
            "tool_call_id": tool_call.id,
            "role": "tool",
            "name": func_name,
            "content": result
        })
else:
    print("没有工具调用行为")

调用: get_location, 参数: {}
结果: 北京
调用: calculator, 参数: {'expression': '2^10'}
结果: 2^10 = 8


## 3.3 将结果返回给LLM生成最终的回答

In [6]:
messages.append(response.choices[0].message)
for tool_result in tool_results:
    messages.append(tool_result)

print(messages)

response_final = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=messages,
    tools=tools
)

reasoning = response_final.choices[0].message.reasoning_content
if reasoning:
    print("=" * 50)
    print("模型推理过程")
    print("=" * 50)
    print(reasoning)
    print()

print("=" * 50)
print("模型最终回答")
print("=" * 50)
print(response_final.choices[0].message.content)

tc = response_final.choices[0].message.tool_calls
if tc:
    print("调用工具:", response_final.choices[0].message.tool_calls)

[{'role': 'system', 'content': '你是一个智能助手，你可以使用工具回答问题。'}, {'role': 'user', 'content': '今天天气怎么样？以及2的10次方是多少？'}, ChatCompletionMessage(content='好的！我先查一下你的所在城市，同时计算2的10次方。', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_3IuDAMS3sKhtpEhvWHXl3654', function=Function(arguments='{}', name='get_location'), type='function', index=0), ChatCompletionMessageFunctionToolCall(id='call_01_GUQhbdegzhqzf2pBa8ff6198', function=Function(arguments='{"expression": "2^10"}', name='calculator'), type='function', index=1)], reasoning_content='用户想知道今天的天气以及2的10次方的计算结果。我需要先获取用户所在城市，然后查天气，同时计算2的10次方。\n\n让我先获取用户所在城市，同时计算2的10次方。这两个任务没有依赖关系，可以同时进行。'), {'tool_call_id': 'call_00_3IuDAMS3sKhtpEhvWHXl3654', 'role': 'tool', 'name': 'get_location', 'content': '北京'}, {'tool_call_id': 'call_01_GUQhbdegzhqzf2pBa8ff6198', 'role': 'tool', 'name': 'calculator', 'content': '2^10 = 8'}]
模型推理过程
看起来计算器可能把 "2^10" 理解成了按位异或运算（XOR），而不是乘方。在数

### 3.4 完整的调用循环

模型根据上下文判断是否需要**继续调用**工具。

In [8]:
def chat_with_tools(user_message, tools, tool_functions):
    messages = [{"role": "system", "content": "你是一个智能助手。"}]
    messages.append({"role": "user", "content": user_message})
    
    for i in range(10):
        print(f"===== Iteration {i+1} =====")
        response = client.chat.completions.create(
            model="deepseek-v4-flash", messages=messages, tools=tools, tool_choice="auto"
        )
        assistant_msg = response.choices[0].message
        
        if not assistant_msg.tool_calls:
            return assistant_msg.content
        
        messages.append(assistant_msg)
        
        for tool_call in assistant_msg.tool_calls:
            func_name = tool_call.function.name
            func_args = eval(tool_call.function.arguments)
            print(f"调用: {func_name}, 参数: {func_args}")
            
            if func_name in tool_functions:
                result = tool_functions[func_name](**func_args) if func_args else tool_functions[func_name]()
            else:
                result = f"错误：工具 {func_name} 不存在"
            
            print(f"结果: {result}")
            messages.append({"tool_call_id": tool_call.id, "role": "tool", "name": func_name, "content": result})

    return "达到最大迭代"

# 测试
print(chat_with_tools("今天天气怎么样？另外 2 的 10 次方是多少？", tools, tool_functions))

===== Iteration 1 =====
调用: get_location, 参数: {}
结果: 汕头
调用: calculator, 参数: {'expression': '2^10'}
结果: 2^10 = 8
===== Iteration 2 =====
调用: calculator, 参数: {'expression': '2**10'}
结果: 2**10 = 1024
===== Iteration 3 =====
调用: get_weather, 参数: {'city': '汕头'}
结果: 汕头: 晴 温度: 28C
===== Iteration 4 =====
好的，下面回答你的两个问题：

### 🌤️ 今天的天气（汕头）
- **天气状况**：☀️ 晴天
- **温度**：28°C

### 🧮 2 的 10 次方
- **结果**：**1024**

所以，你那边今天天气不错，适合出门活动！数学题也解决了，2¹⁰ = **1024**（也就是电脑领域常说的 1K）。还有什么需要帮忙的吗？😊


###langchain版本


In [13]:
import os
import math
from dotenv import load_dotenv
from pathlib import Path

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_deepseek import ChatDeepSeek
from langchain.agents import create_agent

# 1. 读取 .env
env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"

load_dotenv(dotenv_path=env_path, override=True)

api_key = os.getenv("DEEPSEEK_API_KEY")
print("是否读取到 API Key:", bool(api_key))


# 2. 定义工具
@tool
def calculator(expression: str) -> str:
    """
    执行简单数学运算。
    """
    allowed = {
        "sin": math.sin,
        "cos": math.cos,
        "sqrt": math.sqrt,
        "pi": math.pi,
    }

    # 把常见的 ^ 转成 Python 幂运算 **
    expression = expression.replace("^", "**")

    result = eval(expression, {"__builtins__": {}}, allowed)
    return f"{expression} = {result}"


@tool
def get_weather(city: str) -> str:
    """
    根据城市查询天气。
    """
    weather_data = {
        "汕头": {"天气": "晴", "温度": "28°C"},
        "深圳": {"天气": "多云", "温度": "21°C"},
        "上海": {"天气": "阵雨", "温度": "18°C"},
        "北京": {"天气": "晴", "温度": "30°C"},
    }

    if city in weather_data:
        d = weather_data[city]
        return f"{city}的天气是{d['天气']}，温度是{d['温度']}。"

    return "找不到该城市的天气。"


@tool
def get_location() -> str:
    """
    获取用户所在城市。
    """
    return "汕头"


# 3. 创建 LLM
llm = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0,
    api_key=api_key,
)

# 4. 创建 Agent
langchain_tools = [get_location, get_weather, calculator]

agent = create_agent(
    model=llm,
    tools=langchain_tools,
)

print("Agent 创建成功")


# 5. 调用 Agent
result = agent.invoke({
    "messages": [
        SystemMessage(content="你是一个智能助手，你可以使用工具回答问题。"),
        HumanMessage(content="今天天气怎么样？以及2的10次方是多少？")
    ]
})

# 6. 输出最终结果
print(result["messages"][-1].content)

# 7. 打印完整消息过程
for msg in result["messages"]:
    msg.pretty_print()

是否读取到 API Key: True
Agent 创建成功
以下是您想了解的信息：

### 🌤️ 今日天气（汕头）
- **天气状况**：☀️ 晴天
- **温度**：28°C

### 🔢 数学计算
- **2的10次方** = **1024**

如果您在其他城市，也可以告诉我，我帮您查一下当地的天气哦！
================================ System Message ================================

你是一个智能助手，你可以使用工具回答问题。
================================ Human Message =================================

今天天气怎么样？以及2的10次方是多少？
================================== Ai Message ==================================

好的，我先获取您所在的城市，然后查询天气，同时计算2的10次方。
Tool Calls:
  get_location (call_00_Z8ZbB6J0YZQS2MwKsDeX7476)
 Call ID: call_00_Z8ZbB6J0YZQS2MwKsDeX7476
  Args:
  calculator (call_01_BL3WU9QEb3wO5CRSkIOw7341)
 Call ID: call_01_BL3WU9QEb3wO5CRSkIOw7341
  Args:
    expression: 2^10
================================= Tool Message =================================
Name: get_location

汕头
================================= Tool Message =================================
Name: calculator

2**10 = 1024
================================== Ai Message =========================

###2.5简单的联网搜索agent

In [18]:
from dotenv import load_dotenv
load_dotenv()

from langchain_tavily import TavilySearch

tavily = TavilySearch(api_key=os.getenv("TAVILY_API_KEY"), max_results=5)
llm=ChatDeepSeek(
    model="deepseek-v4-flash", 
    temperature=0,
    api_key=os.getenv("DEEPSEEK_API_KEY")
)

agent = create_agent(
    model=llm,
    tools=[ tavily],
)
result=agent.invoke({
    "messages":[
        HumanMessage(content="帮我搜索一下汕头大学的最新新闻，并且告诉我现在的天气")
    ]
})
print(result["messages"][-1].content)

好的，查到了！以下是汕头大学的最新新闻和汕头的天气情况：

---

## 🏫 汕头大学最新新闻

### 1️⃣ 汕头大学举办第二届"510·我要廉"廉洁文化日
📅 **2026年5月12日**
汕头大学东海岸校区举办了第二届廉洁文化日活动，由学校纪委指导，联合法学院、文学院、商学院等多个学院共同开展。活动包括：
- 与汕头市纪委监委合作举办《**侨批中的廉洁文化**》专题展览
- **主题辩论赛、微视频、海报设计、书法、征文**等多类赛事
- 吸引约 **150名师生**参与，打造沉浸式廉洁教育场景

### 2️⃣ 汕头大学2025年公开招聘拟聘人员公示
📅 **2026年5月8日**
汕头大学发布了2025年公开招聘事业单位工作人员的拟聘人员名单公示，公示期为5个工作日。

### 3️⃣ 汕头大学医学院持续招聘人才
医学院及附属医院正在招聘**博士后、骨科、神经学科、甲乳外科、心血管病医院**等多个团队的科研人才。

---

## 🌤️ 汕头今日天气

| 项目 | 内容 |
|------|------|
| **天气状况** | ⛈️ **雷阵雨** |
| **气温范围** | 🌡️ **24°C ~ 31°C** |
| **湿度** | 约 **76%** |
| **风力** | 东南风/西南风 **2~3级** |
| **穿衣建议** | 👕 建议穿着吸湿排汗的夏装、棉麻质地的短打扮 |

> 💡 **温馨提示**：汕头今天有雷阵雨，天气潮湿闷热，出门记得带伞哦！☂️


###MCP


In [14]:
!uv add fastmcp langchain_mcp_adapters

Resolved 140 packages in 10.36s
 Downloaded beartype
 Downloaded cryptography
 Downloaded pywin32
Prepared 48 packages in 1m 38s
Installed 48 packages in 945ms
 + aiofile==3.9.0
 + authlib==1.7.2
 + beartype==0.22.9
 + cachetools==7.1.1
 + caio==0.9.25
 + click==8.3.3
 + cryptography==48.0.0
 + cyclopts==4.12.0
 + dnspython==2.8.0
 + docstring-parser==0.18.0
 + docutils==0.22.4
 + email-validator==2.3.0
 + exceptiongroup==1.3.1
 + fastmcp==3.2.4
 + griffelib==2.0.2
 + httpx-sse==0.4.3
 + importlib-metadata==8.7.1
 + jaraco-classes==3.4.0
 + jaraco-context==6.1.2
 + jaraco-functools==4.4.0
 + joserfc==1.6.5
 + jsonref==1.1.0
 + jsonschema-path==0.4.6
 + keyring==25.7.0
 + langchain-mcp-adapters==0.2.2
 + markdown-it-py==4.2.0
 + mcp==1.27.1
 + mdurl==0.1.2
 + more-itertools==11.0.2
 + openapi-pydantic==0.5.1
 + opentelemetry-api==1.41.1
 + pathable==0.5.0
 + py-key-value-aio==0.4.4
 + pydantic-settings==2.14.1
 + pyjwt==2.12.1
 + pyperclip==1.11.0
 + python-multipart==0.0.28
 + pywin32=

In [ ]:
from mcp.server import FastMcp
import math

#创建一个mcp server
mcp=FastMcp("caoheming")



###
3.1mcp连接一个服务

In [6]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "my_mcp": {
            "transport": "stdio",
            "command": "python",
            "args": [
                r"C:\Users\ASUS\Desktop\llm_jupyter_env\跟敲\main.py",
            ],
        }
    }
)

mcp_tools = await mcp_client.get_tools()

for tool in mcp_tools:
    print(tool.name)

UnsupportedOperation: fileno